# Ontology Lookup Service (OLS) — API & bulk

bioDB exposes the OBO Foundry ecosystem through three complementary
surfaces. EBI's [Ontology Lookup Service](https://www.ebi.ac.uk/ols4/)
is the API gateway (~280 ontologies — Mondo, HPO, EFO, SO, GO, ChEBI,
SNOMED, …). The right surface to reach for depends on whether you
need one term or the whole graph.

| Mode | Module | When to reach for it |
|---|---|---|
| **OLS REST** | `biodb.ols` | One term / a few hops, no local file needed. Targets ~280 ontologies hosted by EMBL-EBI. |
| **OWL entity walk** | `biodb.ontology.get_ontology / get_descendants / get_ancestors / get_mrca` | Local OWL file already in hand — full-fidelity owlready2 graph traversal. |
| **Bulk OBO + helpers** | `biodb.ontology.expand_keyword_sets_from_ontology`, plotting, similarity matrices | Whole-graph offline analysis after you've parsed an OBO/OWL release. |

This notebook drives all three against **real Mondo** so the
behaviour you see is what production code returns.

## 1. API mode — OLS REST against live Mondo

No local files required. `biodb.ols` wraps the [EBI OLS4 REST
API](https://www.ebi.ac.uk/ols4/api-docs) — every helper accepts a
CURIE (`MONDO:0004975`) or a full IRI.

In [1]:
from biodb import ols

mondo = ols.get_ontology("mondo")
{k: mondo[k] for k in ("ontologyId", "version", "numberOfTerms", "loaded")}

{'ontologyId': 'mondo',
 'version': '2026-05-05',
 'numberOfTerms': 58940,
 'loaded': '2026-05-18T00:08:39.523402120'}

Pull a single term — full record with synonyms, definition, OBO ID,
IRI, obsolescence flag.

In [2]:
ad = ols.get_term("mondo", "MONDO:0004975")
{
    "obo_id": ad["obo_id"],
    "label": ad["label"],
    "synonyms": ad["synonyms"][:3],
    "description": (ad["description"] or [""])[0][:120] + "…",
    "is_obsolete": ad["is_obsolete"],
}

{'obo_id': 'MONDO:0004975',
 'label': 'Alzheimer disease',
 'synonyms': ['Alzheimer dementia',
  'Alzheimers disease',
  'presenile and senile dementia'],
 'description': 'A progressive, neurodegenerative disease characterized by loss of function and death of nerve cells in several areas of …',
 'is_obsolete': False}

Walk the full descendant subtree of *Alzheimer disease*. Pagination
is handled internally; for very large subtrees (e.g. the GO root) it
transparently chases the HAL `_links.next.href` chain.

In [3]:
descendants = ols.get_descendants("mondo", "MONDO:0004975")
print(f"{len(descendants)} descendants")
descendants[["obo_id", "label"]].head(10)

21 descendants


,obo_id,label
0,MONDO:0100087,familial Alzheimer disease
1,MONDO:0014316,Alzheimer disease 19
2,MONDO:0014265,Alzheimer disease 18
3,MONDO:0014036,Alzheimer disease 17
4,MONDO:0010422,Alzheimer disease 16
5,MONDO:0015140,early-onset autosomal dominant Alzheimer disease
6,MONDO:0007089,Alzheimer disease 2
7,MONDO:0012631,Alzheimer disease 14
8,MONDO:0012630,Alzheimer disease 13
9,MONDO:0012609,Alzheimer disease 12


Ancestors expose the upward closure — every super-class up to the
Mondo roots.

In [4]:
ancestors = ols.get_ancestors("mondo", "MONDO:0004975")
ancestors[["obo_id", "label"]].head(8)

,obo_id,label
0,MONDO:0005574,tauopathy
1,MONDO:0001627,dementia
2,MONDO:0005559,neurodegenerative disease
3,MONDO:7770011,disease by molecular mechanism
4,MONDO:0002039,cognitive disorder
5,MONDO:0002602,central nervous system disorder
6,MONDO:7770007,disease by developmental or physiological process
7,MONDO:0002025,psychiatric disorder


`search` is the Solr-backed full-text endpoint — handy when you have
a label or synonym but no CURIE.

In [5]:
hits = ols.search("alzheimer", ontology="mondo", rows=5)
hits[["obo_id", "label"]]

,obo_id,label
0,MONDO:0004975,Alzheimer disease
1,MONDO:0100087,familial Alzheimer disease
2,MONDO:1011460,"Alzheimer disease, degu"
3,MONDO:1011461,"Alzheimer disease, dog"
4,MONDO:1011463,"Alzheimer disease, sheep"


## 2. OWL entity walk (when you already have an OWL file)

`biodb.ontology.get_ontology` loads any OBO Foundry OWL file via
`owlready2`. The returned object exposes every class — you can walk it
freely with `get_descendants` / `get_ancestors` / `get_mrca`.

(Not executed here — the Mondo OWL is ~30 MB and owlready2 builds an
in-memory triple store on first load.)

```python
from biodb import ontology

mondo = ontology.get_ontology(
    "http://purl.obolibrary.org/obo/mondo.owl"
).load()
alzheimers = mondo.search_one(iri="*MONDO_0004975")
descendants = ontology.get_descendants(alzheimers)
```

## 3. Bulk — N-hop keyword expansion

Once you have a `{parent: [children]}` dict (typically built from a
parsed Mondo / HPO release, or — as below — straight from OLS), the
`expand_keyword_sets_from_ontology` helper does N-hop seed expansion.
This is the cheap-and-cheerful way to turn one disease term into a
search-friendly synonym list.

In [6]:
from biodb import ontology

# Build a tiny in-memory subgraph from the OLS descendant table.
graph = {ad["label"]: descendants["label"].head(4).tolist()}
for label in descendants["label"].head(4):
    graph[label] = []

ontology.expand_keyword_sets_from_ontology(
    seed_keywords={"alzheimer": [ad["label"]]},
    ontology_dict=graph,
    n_hops=1,
)

{'alzheimer': ['Alzheimer disease',
  'Alzheimer disease 17',
  'Alzheimer disease 18',
  'Alzheimer disease 19',
  'familial Alzheimer disease']}

For real ontology-similarity work — Lin/Resnik scores, ontology
matrix builders, attention-weight analysis — see the
`biodb.ontology.*` helpers; they operate on the parsed OBO release
in `~/.cache/biodb/ontology/`.